# Train the ClimateTwin regional models

Train one scikit-learn linear regression per Indian state or Union Territory from the daily synthetic climate data. The fitted models predict the monthly average low, monthly average high, and total rainfall.

Open this notebook from a local copy of the [ClimateTwin repository](https://github.com/Rishupatel-official/ananta) in Jupyter or VS Code. Use a Python 3.12 kernel with the packages in `model/requirements.txt` installed and select **Run All**. The CSV is already in the repository. The code cells show the training, test scores, and an example prediction from the saved model.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Jupyter can start in either the repository root or the notebooks folder.
project = next((p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "model" / "train.py").is_file()
                and (p / "notebooks" / "climate_daily.csv").is_file()), None)
if project is None:
    raise FileNotFoundError("Open the notebook from the cloned ClimateTwin repository.")
sys.path.insert(0, str(project))

from model.train import FEATURES, TARGETS, aggregate_monthly, features, metrics, postprocess
from model.inference import ClimateModel, save_bundle

## 1. Turn daily weather into monthly training examples

The project's preparation function averages daily lows and highs, sums rainfall, and keeps months that have all three measurements on every date. The two temperature targets are measured in °C and rainfall in mm per month.

In [2]:
csv_path = project / "notebooks" / "climate_daily.csv"
daily = pd.read_csv(csv_path, usecols=["date", "state", *TARGETS])
monthly, audit = aggregate_monthly(daily)
first_month, last_month = monthly.month.min(), monthly.month.max()
base_year = first_month.year
test_start = last_month - 23
print(f"{len(daily):,} daily records → {len(monthly):,} complete region-months")
print(f"{audit['stateCount']} regions; {audit['excludedStateMonths']} incomplete region-months excluded")
monthly.loc[monthly.state.eq("Delhi"), ["month", *TARGETS]].head(3)

223,524 daily records → 7,018 complete region-months
36 regions; 326 incomplete region-months excluded


,month,min_temp,max_temp,rainfall
1565,2010-01,12.596774,25.922581,0.0
1566,2010-02,17.382143,30.821429,0.0
1567,2010-04,27.536667,41.383333,4.0


## 2. Train on earlier months and test on later months

For each region, fit an ordinary least squares regression on 2010–2024. Its inputs are the time elapsed since 2010 and 11 calendar-month indicators. January is the reference month. Score the fitted model on 2025–2026, alongside a baseline that predicts each calendar month's historical average for that region.

In [3]:
regions = monthly.groupby("state", sort=True)
evaluations = {}
actual_rows, model_rows, baseline_rows = [], [], []

for state, rows in regions:
    training = rows.loc[rows.month < test_start]
    testing = rows.loc[rows.month >= test_start]

    # Fit using only dates before the test period.
    estimator = LinearRegression()
    estimator.fit(features(training.month, base_year), training[TARGETS].to_numpy())

    raw = estimator.predict(features(testing.month, base_year))
    predicted = postprocess(raw)
    actual = testing[TARGETS].to_numpy()
    calendar_average = training.groupby(training.month.dt.month)[TARGETS].mean()
    baseline = calendar_average.loc[testing.month.dt.month].to_numpy()

    evaluations[state] = {
        "trainMonths": len(training), "testMonths": len(testing),
        "trainFirst": str(training.month.min()), "trainLast": str(training.month.max()),
        "testFirst": str(testing.month.min()), "testLast": str(testing.month.max()),
        "model": metrics(actual, predicted), "baseline": metrics(actual, baseline),
        "clippedRainMonths": int((raw[:, 2] < 0).sum()),
        "reorderedTemperatureMonths": int((raw[:, 0] > raw[:, 1]).sum()),
    }
    actual_rows.append(actual)
    model_rows.append(predicted)
    baseline_rows.append(baseline)

observed = np.concatenate(actual_rows)
pooled = {
    "model": metrics(observed, np.concatenate(model_rows)),
    "baseline": metrics(observed, np.concatenate(baseline_rows)),
}
print(f"Tested {len(observed)} region-months from {test_start} through {last_month}.")

Tested 828 region-months from 2025-01 through 2026-12.


In [4]:
labels = {
    "min_temp": "Average low (°C)", "max_temp": "Average high (°C)",
    "mean_temp": "Mean temperature (°C)", "rainfall": "Monthly rainfall (mm)",
}
scores = pd.DataFrame({
    "Regression MAE": {label: pooled["model"][key]["mae"] for key, label in labels.items()},
    "Seasonal average MAE": {label: pooled["baseline"][key]["mae"] for key, label in labels.items()},
})
scores.round(3)

,Regression MAE,Seasonal average MAE
Average low (°C),0.178,0.228
Average high (°C),0.231,0.270
Mean temperature (°C),0.142,0.205
Monthly rainfall (mm),40.753,39.899


MAE is the average size of the error, so smaller values are better. On the supplied synthetic data, the regression improves temperature estimates. The seasonal average has a slightly smaller rainfall error.

## 3. Train the final 36 models

After measuring test performance, fit a fresh regression for each region using all its complete months from 2010–2026. The `fit` call below produces the models saved for inference. Each one learns separate coefficients for the three targets.

In [5]:
fitted_models = {}
state_metadata = {}

for state, rows in regions:
    X = features(rows.month, base_year)
    y = rows[TARGETS].to_numpy()
    model = LinearRegression()
    model.fit(X, y)
    fitted_models[state] = model

    state_metadata[state] = {
        "intercepts": model.intercept_.tolist(),
        "coefficients": model.coef_.tolist(),
        "fitMonths": len(rows),
        "fitFirst": str(rows.month.min()), "fitLast": str(rows.month.max()),
        "evaluation": evaluations[state],
    }

forecast_months = pd.period_range(last_month + 1, periods=12, freq="M")
print(f"Trained {len(fitted_models)} regional models with {X.shape[1]} features and {y.shape[1]} targets each.")
print(f"Projection period: {forecast_months[0]} through {forecast_months[-1]}")

Trained 36 regional models with 12 features and 3 targets each.
Projection period: 2027-01 through 2027-12


## 4. Save the model and reload it

The notebook writes its own `training_output` directory alongside the notebook. `model.pkl` contains the fitted estimators. `model.info.json` records the checksum and library versions. `regression-model.json` contains the corresponding dashboard metadata. This run records the CSV's checksum; keep these three outputs together if using this run for deployment.

In [6]:
with csv_path.open("rb") as stream:
    input_digest = hashlib.file_digest(stream, "sha256").hexdigest()

metadata = {
    "schemaVersion": 1, "algorithm": "Ordinary least squares: trend + month indicators",
    "dataKind": "synthetic", "inputSha256": input_digest,
    "baseYear": base_year, "featureOrder": FEATURES, "targetOrder": TARGETS,
    "trainingStart": str(first_month), "trainingEnd": str(last_month),
    "forecastStart": str(forecast_months[0]), "forecastEnd": str(forecast_months[-1]),
    "evaluationStart": str(test_start), "evaluationEnd": str(last_month),
    "evaluationMonths": len(observed), "pooledEvaluation": pooled,
    "audit": {key: value for key, value in audit.items() if key != "excludedMonths"},
    "states": state_metadata,
}
output_dir = project / "notebooks" / "training_output"
model_path = output_dir / "model.pkl"
save_bundle(model_path, metadata, fitted_models)
(output_dir / "regression-model.json").write_text(
    json.dumps(metadata, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
reloaded = ClimateModel.load(model_path)
print(f"Saved and reloaded {len(reloaded.models)} models from notebooks/training_output/model.pkl.")

Saved and reloaded 36 models from notebooks/training_output/model.pkl.


In [7]:
# Use the reloaded model for one region and month.
projection = reloaded.predict("Delhi", str(forecast_months[6]))
pd.DataFrame([projection]).round(2)

,state,month,minTemp,maxTemp,meanTemp,rainfall
0,Delhi,2027-07,26.37,40.02,33.2,341.48
